# Preprocessing Pipeline

Cleaning and feature logic only — no EDA. Paired to `scripts/preprocessing.py` via Jupytext, which `train.py` and `explore.ipynb` both import from.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## Transformers

In [ ]:
# drop columns with no predictive value
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

In [ ]:
# pull the honorific out of Name ("Braund, Mr. Owen Harris" -> "Mr"). Title is a working
# column, not a feature: impute_age groups on it and drop_title removes it before one-hot
class TitleAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        title = (
            X["Name"]
            .str.extract(r",\s*([^\.]+)\.", expand=False)
            .str.strip()
            .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        )
        # Dr/Rev/Col/Lady/... are ~2% of rows between them, too thin to fit individually
        X["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
        return X

In [ ]:
# map Sex from text to 0/1
class SexEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["Sex"] = X["Sex"].map({"female": 0, "male": 1})
        return X

In [ ]:
# replace Cabin with a 0/1 "was a cabin recorded" flag -- the value is 77% missing, but
# explore.ipynb shows the missingness itself carries signal (70% vs 29% survival, and it
# holds inside every Pclass, p=0.001 controlling for Pclass, Sex and Fare)
class CabinFlagEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["HasCabin"] = X["Cabin"].notna().astype(int)
        return X.drop(columns=["Cabin"])

In [ ]:
# fill target_col with a per-group stat; fit() sees only the rows it's given,
# so inside a CV fold it never learns from that fold's held-out rows
class GroupStatImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_cols, target_col, stat):
        self.group_cols = group_cols
        self.target_col = target_col
        self.stat = stat

    def fit(self, X, y=None):
        self.group_stats_ = (
            X.groupby(self.group_cols)[self.target_col]
            .agg(self.stat)
            .rename("_fill_value")
            .reset_index()
        )
        # a group can be all-NaN, or absent from the fold we fit on -- both leave the merge
        # below with nothing to fill from, so keep an ungrouped stat as the backstop
        self.global_stat_ = X[self.target_col].agg(self.stat)
        return self

    def transform(self, X):
        merged = X.merge(self.group_stats_, on=self.group_cols, how="left")
        merged[self.target_col] = merged[self.target_col].fillna(merged["_fill_value"])
        merged[self.target_col] = merged[self.target_col].fillna(self.global_stat_)
        return merged.drop(columns=["_fill_value"])

In [ ]:
# flag the one Sex x Pclass interaction that tested significant in explore.ipynb
class InteractionFeatureAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["Male_and_3rdClass"] = ((X["Sex"] == 1) & (X["Pclass"] == 3)).astype(int)
        return X

In [ ]:
# flag boys -- "Master" was the period's honorific for a child male. explore_remaining.ipynb
# has this beating Under10 head to head: it reads the name rather than the age, so it still
# catches a boy whose Age was missing and got imputed to the adult median
class MasterFlagAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["IsMaster"] = (X["Title"] == "Master").astype(int)
        return X

In [ ]:
# flag passengers under 10 -- the age histogram in explore.ipynb shows a survival spike there
class ChildFlagAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["Under10"] = (X["Age"] < 10).astype(int)
        return X

In [ ]:
# bucket family size (SibSp + Parch + self) into alone / 2-4 / 5+ -- survival by family size
# is an inverted U in explore.ipynb, so a linear term cancels out where the buckets don't.
# alone is the reference level: both flags are 0 for it
class FamilyGroupAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        family_size = X["SibSp"] + X["Parch"] + 1
        X["FamilySmall"] = family_size.between(2, 4).astype(int)
        X["FamilyLarge"] = (family_size >= 5).astype(int)
        return X

## Pipeline

In [ ]:
# engineered features a config can switch on by name via its `features` list
OPTIONAL_FEATURES = {
    "male_x_3rdclass": InteractionFeatureAdder,
    "is_master": MasterFlagAdder,
    "under10": ChildFlagAdder,
    "family_group": FamilyGroupAdder,
}

In [ ]:
# raw DataFrame -> model-ready features; safe to hand straight to cross_validate()
def build_preprocessing_pipeline(features=()):
    unknown = set(features) - set(OPTIONAL_FEATURES)
    if unknown:
        raise ValueError(
            f"unknown feature(s) {sorted(unknown)}; valid: {sorted(OPTIONAL_FEATURES)}"
        )

    one_hot = ColumnTransformer(
        transformers=[
            (
                "one_hot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ["Pclass", "Embarked"],
            ),
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    one_hot.set_output(transform="pandas")

    steps = [
        # before drop_columns, which is where Name goes
        ("add_title", TitleAdder()),
        (
            "drop_columns",
            ColumnDropper(["PassengerId", "Name", "Ticket"]),
        ),
        ("encode_sex", SexEncoder()),
        ("encode_cabin", CabinFlagEncoder()),
        (
            "impute_age",
            # grouping on Title instead of Sex ages the missing-age boys correctly (~4 rather
            # than ~26), but explore_remaining.ipynb shows it costs accuracy once is_master is
            # on -- the imputed age then duplicates the flag and the trees lose a split
            GroupStatImputer(
                group_cols=["Pclass", "Sex"], target_col="Age", stat="median"
            ),
        ),
        (
            "impute_embarked",
            GroupStatImputer(
                group_cols=["Pclass"],
                target_col="Embarked",
                stat=lambda s: s.mode().iloc[0],
            ),
        ),
        (
            # train.csv has no gaps here, but test.csv has one -- and a NaN reaching
            # LogisticRegression is an exception rather than a bad prediction
            "impute_fare",
            GroupStatImputer(
                group_cols=["Pclass"], target_col="Fare", stat="median"
            ),
        ),
    ]
    # after imputation so Age is already filled, before one-hot so the new columns pass through
    steps += [(f"add_{name}", OPTIONAL_FEATURES[name]()) for name in features]
    # Title has done its work by here -- as a string column it would survive one-hot's
    # passthrough and hand the model text
    steps.append(("drop_title", ColumnDropper(["Title"])))
    steps.append(("one_hot_encode", one_hot))

    return Pipeline(steps)